# Qwen3-4B trên Kaggle với GPU T4 ×2

1. Nhập tệp `.ipynb` vào Kaggle Notebooks.
2. Trong **Settings**, chọn **Accelerator → GPU T4 x2** và bật **Internet** để tải thư viện và trọng số.
3. Chọn **Run All**. Đổi `prompt` ở ô cuối để thử câu hỏi khác.

Notebook tải [Qwen/Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) bằng Transformers, dùng FP16 và chia các lớp của một bản model lên hai GPU. Cách này chia bộ nhớ cần cho trọng số. Tốc độ trả lời một câu còn phụ thuộc vào việc truyền dữ liệu giữa hai GPU.

In [ ]:
import torch

if torch.cuda.device_count() != 2:
    raise RuntimeError("Trong Settings của Kaggle, chọn Accelerator → GPU T4 x2 rồi chạy lại.")

for gpu_id in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(gpu_id)
    print(f"GPU {gpu_id} · {properties.name} · {properties.total_memory / 1024**3:.1f} GiB")


In [ ]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0" fastapi uvicorn


In [ ]:
from collections import Counter

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="balanced",
    attn_implementation="sdpa",
).eval()

parameter_bytes: Counter[torch.device] = Counter()
for parameter in model.parameters():
    parameter_bytes[parameter.device] += parameter.numel() * parameter.element_size()

for device, size in sorted(parameter_bytes.items(), key=lambda entry: str(entry[0])):
    print(f"{device} · {size / 1024**3:.2f} GiB trọng số")

if set(parameter_bytes) != {torch.device("cuda:0"), torch.device("cuda:1")}:
    raise RuntimeError(f"Trọng số phải nằm trên cả hai GPU. Phân bổ hiện tại: {model.hf_device_map}")

input_device = model.get_input_embeddings().weight.device


In [ ]:
@torch.inference_mode()
def generate_reply(messages: list[dict[str, str]], max_new_tokens: int = 256) -> tuple[str, int, int, str]:
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        formatted_prompt, return_tensors="pt", add_special_tokens=False
    ).to(input_device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
    )
    prompt_tokens = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_tokens:]
    eos = model.generation_config.eos_token_id
    eos_ids = eos if isinstance(eos, list) else [eos]
    finish_reason = "stop" if generated_ids[-1].item() in eos_ids else "length"
    reply = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return reply, prompt_tokens, len(generated_ids), finish_reason


prompt = "Giải thích bằng tiếng Việt, trong 3 câu, vì sao bầu trời có màu xanh."
print(generate_reply([{"role": "user", "content": prompt}])[0])


## Gọi API từ bên ngoài Kaggle

Giữ notebook ở chế độ riêng tư vì ô khởi động hiển thị `LLM_TOKEN`. Sau khi khởi động API, sao chép cặp `LLM_URL` và `LLM_TOKEN` từ output hiện tại. URL hoạt động trong thời gian server, tunnel và phiên Kaggle còn chạy.

API hỗ trợ `GET /v1/models` và `POST /v1/chat/completions` với tin nhắn văn bản và `stream: false`. Token được gửi qua header `Authorization: Bearer ...`.

In [ ]:
import secrets
import time
from threading import Lock
from typing import Annotated, Literal

from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, Field

API_MODEL = "qwen3:4b"
LLM_TOKEN = secrets.token_urlsafe(32)
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
bearer = HTTPBearer(auto_error=False)
generation_lock = Lock()


def require_token(
    credentials: Annotated[HTTPAuthorizationCredentials | None, Depends(bearer)],
) -> None:
    if credentials is None or not secrets.compare_digest(
        credentials.credentials.encode("utf-8"), LLM_TOKEN.encode("ascii")
    ):
        raise HTTPException(401, "Token không hợp lệ", headers={"WWW-Authenticate": "Bearer"})


class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=16_000)


class ChatRequest(BaseModel):
    model: Literal["qwen3:4b", "Qwen/Qwen3-4B"]
    messages: list[Message] = Field(min_length=1, max_length=32)
    max_tokens: int = Field(default=256, ge=1, le=2048)
    stream: Literal[False] = False


@app.get("/v1/models", dependencies=[Depends(require_token)])
def list_models() -> dict:
    return {"object": "list", "data": [
        {"id": API_MODEL, "object": "model", "created": 0, "owned_by": "qwen"}
    ]}


@app.post("/v1/chat/completions", dependencies=[Depends(require_token)])
def chat_completions(request: ChatRequest) -> dict:
    if sum(len(message.content) for message in request.messages) > 16_000:
        raise HTTPException(400, "Tổng nội dung vượt quá 16000 ký tự")
    if not generation_lock.acquire(blocking=False):
        raise HTTPException(429, "Model đang xử lý yêu cầu khác", headers={"Retry-After": "2"})
    try:
        messages = [message.model_dump() for message in request.messages]
        reply, prompt_tokens, completion_tokens, finish_reason = generate_reply(
            messages, request.max_tokens
        )
    finally:
        generation_lock.release()
    return {
        "id": "chatcmpl-" + secrets.token_hex(12),
        "object": "chat.completion",
        "created": int(time.time()),
        "model": API_MODEL,
        "choices": [
            {"index": 0, "message": {"role": "assistant", "content": reply},
             "finish_reason": finish_reason}
        ],
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
        },
    }


In [ ]:
import re
import subprocess
from pathlib import Path
from threading import Thread
from urllib.error import URLError
from urllib.request import Request, urlopen

import uvicorn


def stop_api() -> None:
    if tunnel is not None and tunnel.poll() is None:
        tunnel.terminate()
        try:
            tunnel.wait(timeout=5)
        except subprocess.TimeoutExpired:
            tunnel.kill()
            tunnel.wait()
    if server is not None:
        server.should_exit = True
    if server_thread is not None:
        server_thread.join(timeout=10)


def wait_for_models(base_url: str, timeout: int) -> None:
    deadline = time.monotonic() + timeout
    request = Request(
        base_url + "/models",
        headers={"Authorization": f"Bearer {LLM_TOKEN}"},
    )
    while True:
        try:
            with urlopen(request, timeout=4) as response:
                if response.status == 200:
                    return
        except (URLError, TimeoutError) as error:
            if time.monotonic() >= deadline:
                raise RuntimeError("API chưa sẵn sàng tại " + base_url) from error
        time.sleep(1)


tunnel = globals().get("tunnel")
server = globals().get("server")
server_thread = globals().get("server_thread")
stop_api()

binary = Path("/tmp/cloudflared")
if not binary.exists():
    download = binary.with_suffix(".download")
    subprocess.run([
        "curl", "-fsSL", "--retry", "3", "--max-time", "120",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", str(download),
    ], check=True)
    download.chmod(0o700)
    download.replace(binary)

PORT = 8000
local_url = f"http://127.0.0.1:{PORT}"
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, access_log=False))
server_thread = Thread(target=server.run, daemon=True)
server_thread.start()

log_path = Path("/tmp/qwen-cloudflared.log")
tunnel = None
try:
    wait_for_models(local_url + "/v1", timeout=15)
    with log_path.open("w") as log:
        tunnel = subprocess.Popen([
            str(binary), "tunnel", "--no-autoupdate", "--protocol", "http2",
            "--url", local_url,
        ], stdout=log, stderr=subprocess.STDOUT)

    deadline = time.monotonic() + 60
    while tunnel.poll() is None and time.monotonic() < deadline:
        match = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", log_path.read_text())
        if match:
            LLM_URL = match.group(0) + "/v1"
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"Cloudflare chưa cấp URL. Xem log tại {log_path}.")

    wait_for_models(LLM_URL, timeout=40)
except BaseException:
    stop_api()
    raise

print(f"LLM_URL={LLM_URL}")
print(f"LLM_TOKEN={LLM_TOKEN}")
print(f"LLM_MODEL={API_MODEL}")


## Ví dụ gửi request từ máy khác

Thay hai giá trị mẫu bằng output của ô trước. `LLM_URL` đã có hậu tố `/v1`.

```python
import requests

llm_url = "https://THAY-URL.trycloudflare.com/v1"
llm_token = "THAY_TOKEN"

response = requests.post(
    f"{llm_url}/chat/completions",
    headers={"Authorization": f"Bearer {llm_token}"},
    json={
        "model": "qwen3:4b",
        "messages": [{"role": "user", "content": "Chào bạn bằng tiếng Việt."}],
        "max_tokens": 128,
        "stream": False,
    },
    timeout=120,
)
response.raise_for_status()
print(response.json()["choices"][0]["message"]["content"])
```

Nếu dùng OpenAI SDK, đặt `base_url=llm_url` và `api_key=llm_token`. Để tắt API, chạy `stop_api()` trong một ô mới.